In [ ]:
#Avaliação Contexto Fixo Deepseek
import pandas as pd
import requests
import json
import time
import re


API_KEY = "

df_gabarito = pd.read_csv("perguntas_respostas_unificadas.xls")
df_respostas = pd.read_csv("resultados_fixo_50chunks_deepseek.csv")

# Limpeza
df_gabarito["resposta"] = df_gabarito["resposta"].fillna("").astype(str)
df_respostas["resposta"] = df_respostas["resposta"].fillna("").astype(str)
df_gabarito["pergunta"] = df_gabarito["pergunta"].fillna("").astype(str)

# Garantir mesmo tamanho
n = min(len(df_gabarito), len(df_respostas))

perguntas = df_gabarito["pergunta"].iloc[:n].tolist()
referencias = df_gabarito["resposta"].iloc[:n].tolist()
geradas = df_respostas["resposta"].iloc[:n].tolist()


def extrair_json(texto):
    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if match:
        return match.group(0)
    return None

def avaliar_com_gemini(pergunta, gabarito, resposta, idx):
    
    prompt = f"""
    You are an expert evaluator for question answering systems.

    Your task is to evaluate a generated answer by comparing it with a reference answer.

    Question:
    {pergunta}

    Reference Answer:
    {gabarito}

    Generated Answer:
    {resposta}

    Evaluate the generated answer based on the following criteria:

    1. Correctness:
    Is the generated answer factually correct according to the reference answer?

    2. Completeness:
    Does the generated answer cover all the important information from the reference answer?

    3. Relevance:
    Does the generated answer actually answer the question?

    4. Consistency:
    Does the generated answer avoid contradictions with the reference answer?

    For each criterion, assign a score from 1 (very poor) to 5 (excellent).

    Then compute the final_score as the average of the four scores.

    Return ONLY a valid JSON in the following format:

    {{
      "correctness": number,
      "completeness": number,
      "relevance": number,
      "consistency": number,
      "final_score": number,
      "justification": "brief explanation"
    }}
    """

    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:generateContent?key={API_KEY}"

    payload = {
        "contents": [
            {
                "parts": [{"text": prompt}]
            }
        ],
        "generationConfig": {
            "temperature": 0
        }
    }

    try:
        print(f"[{idx}] Enviando requisição...")

        response = requests.post(url, json=payload)

        print(f"[{idx}] Status HTTP: {response.status_code}")

        if response.status_code != 200:
            print(f"[{idx}] ERRO HTTP: {response.text}")
            return None

        result = response.json()

        text = result["candidates"][0]["content"]["parts"][0]["text"]

        print(f"[{idx}] Resposta recebida (primeiros 100 chars): {text[:100]}")

        json_str = extrair_json(text)

        if json_str:
            return json.loads(json_str)
        else:
            print(f"[{idx}] JSON não encontrado!")
            return None

    except Exception as e:
        print(f"[{idx}] ERRO no parsing: {e}")
        return None
        
# Avaliar todas
resultados = []

for i in range(n):

    print(f"\n===== [{i+1}/{n}] =====")

    pergunta = perguntas[i]
    gabarito = referencias[i]
    resposta = geradas[i]

    print("❓ Pergunta:", pergunta)
    print("📘 Gabarito:", gabarito[:200])
    print("🤖 Gerada:", resposta[:200])

    avaliacao = avaliar_com_gemini(
        pergunta,
        gabarito,
        resposta,
        i
    )

    if avaliacao:
        resultados.append({
            "idx": i,
            "pergunta": pergunta,
            "resposta_gabarito": gabarito,
            "resposta_gerada": resposta,
            **avaliacao
        })
        print("✅ OK")
    else:
        print("❌ Falhou")

    time.sleep(2)
    
df_result = pd.DataFrame(resultados)
df_result.to_csv("avaliacao2_Fixo_llm_deepseek.csv", index=False)



===== [1/192] =====
❓ Pergunta: Quais formulários são mencionados no documento como disponíveis no site do HE -> Sistemas -> ADS Hospitalar -> Documentos -> USOST – Unidade de Saúde Ocupacional e Segurança do Trabalho, e para quais finalidades são utilizados?
📘 Gabarito: Os formulários mencionados são o USOST-038 - Controle de acesso em espaço confinado, o USOST-028 - Inspeção de trabalho em altura, o USOST-029 - Autorização para trabalho a quente e o USOST-030 - Perm
🤖 Gerada: Os formulários mencionados como disponíveis no site do HE -> Sistemas -> ADS Hospitalar -> Documentos -> USOST são:

- **USOST-027 – Permissão de Trabalho de Risco (PTR)**: utilizado para autorizar at
[0] Enviando requisição...
[0] Status HTTP: 200
[0] Resposta recebida (primeiros 100 chars): ```json
{
  "correctness": 5,
  "completeness": 5,
  "relevance": 5,
  "consistency": 5,
  "final_sc
✅ OK

===== [2/192] =====
❓ Pergunta: De acordo com o documento, qual é a definição de Estágio Optativo para acadêmicos d

In [2]:
#Avaliação Incremental Deepseek
import pandas as pd
import requests
import json
import time
import re


API_KEY = "

df_gabarito = pd.read_csv("perguntas_respostas_unificadas.xls")
df_respostas = pd.read_csv("resultados_incremental_deepseek.csv")

# Limpeza
df_gabarito["resposta"] = df_gabarito["resposta"].fillna("").astype(str)
df_respostas["resposta"] = df_respostas["resposta"].fillna("").astype(str)
df_gabarito["pergunta"] = df_gabarito["pergunta"].fillna("").astype(str)

# Garantir mesmo tamanho
n = min(len(df_gabarito), len(df_respostas))

perguntas = df_gabarito["pergunta"].iloc[:n].tolist()
referencias = df_gabarito["resposta"].iloc[:n].tolist()
geradas = df_respostas["resposta"].iloc[:n].tolist()


def extrair_json(texto):
    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if match:
        return match.group(0)
    return None

def avaliar_com_gemini(pergunta, gabarito, resposta, idx):
    
    prompt = f"""
    You are an expert evaluator for question answering systems.

    Your task is to evaluate a generated answer by comparing it with a reference answer.

    Question:
    {pergunta}

    Reference Answer:
    {gabarito}

    Generated Answer:
    {resposta}

    Evaluate the generated answer based on the following criteria:

    1. Correctness:
    Is the generated answer factually correct according to the reference answer?

    2. Completeness:
    Does the generated answer cover all the important information from the reference answer?

    3. Relevance:
    Does the generated answer actually answer the question?

    4. Consistency:
    Does the generated answer avoid contradictions with the reference answer?

    For each criterion, assign a score from 1 (very poor) to 5 (excellent).

    Then compute the final_score as the average of the four scores.

    Return ONLY a valid JSON in the following format:

    {{
      "correctness": number,
      "completeness": number,
      "relevance": number,
      "consistency": number,
      "final_score": number,
      "justification": "brief explanation"
    }}
    """

    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-pro:generateContent?key={API_KEY}"

    payload = {
        "contents": [
            {
                "parts": [{"text": prompt}]
            }
        ],
        "generationConfig": {
            "temperature": 0
        }
    }

    try:
        print(f"[{idx}] Enviando requisição...")

        response = requests.post(url, json=payload)

        print(f"[{idx}] Status HTTP: {response.status_code}")

        if response.status_code != 200:
            print(f"[{idx}] ERRO HTTP: {response.text}")
            return None

        result = response.json()

        text = result["candidates"][0]["content"]["parts"][0]["text"]

        print(f"[{idx}] Resposta recebida (primeiros 100 chars): {text[:100]}")

        json_str = extrair_json(text)

        if json_str:
            return json.loads(json_str)
        else:
            print(f"[{idx}] JSON não encontrado!")
            return None

    except Exception as e:
        print(f"[{idx}] ERRO no parsing: {e}")
        return None
        
# Avaliar todas
resultados = []

for i in range(n):

    print(f"\n===== [{i+1}/{n}] =====")

    pergunta = perguntas[i]
    gabarito = referencias[i]
    resposta = geradas[i]

    print("❓ Pergunta:", pergunta)
    print("📘 Gabarito:", gabarito[:200])
    print("🤖 Gerada:", resposta[:200])

    avaliacao = avaliar_com_gemini(
        pergunta,
        gabarito,
        resposta,
        i
    )

    if avaliacao:
        resultados.append({
            "idx": i,
            "pergunta": pergunta,
            "resposta_gabarito": gabarito,
            "resposta_gerada": resposta,
            **avaliacao
        })
        print("✅ OK")
    else:
        print("❌ Falhou")

    time.sleep(2)
    
df_result = pd.DataFrame(resultados)
df_result.to_csv("avaliacao_Incremental_llm_deepseek.csv", index=False)



===== [1/192] =====
❓ Pergunta: Quais formulários são mencionados no documento como disponíveis no site do HE -> Sistemas -> ADS Hospitalar -> Documentos -> USOST – Unidade de Saúde Ocupacional e Segurança do Trabalho, e para quais finalidades são utilizados?
📘 Gabarito: Os formulários mencionados são o USOST-038 - Controle de acesso em espaço confinado, o USOST-028 - Inspeção de trabalho em altura, o USOST-029 - Autorização para trabalho a quente e o USOST-030 - Perm
🤖 Gerada: No site do HE (Sistemas -> ADS Hospitalar -> Documentos -> USOST) estão disponíveis os formulários USOST-027 (Permissão de Trabalho de Risco - PTR) para atividades de risco não rotineiras ou previamen
[0] Enviando requisição...
[0] Status HTTP: 200
[0] Resposta recebida (primeiros 100 chars): ```json
{
  "correctness": 2,
  "completeness": 2,
  "relevance": 5,
  "consistency": 1,
  "final_sc
✅ OK

===== [2/192] =====
❓ Pergunta: De acordo com o documento, qual é a definição de Estágio Optativo para acadêmicos d

In [3]:
import pandas as pd

df_fixo = pd.read_csv("avaliacao2_Fixo_llm_deepseek.csv")
df_inc = pd.read_csv("avaliacao_Incremental_llm_deepseek.csv")

cols = ["correctness","completeness","relevance","consistency","final_score"]

for col in cols:
    df_fixo[col] = pd.to_numeric(df_fixo[col], errors="coerce")
    df_inc[col] = pd.to_numeric(df_inc[col], errors="coerce")

print(df_fixo.dtypes)
print(df_inc.dtypes)

# Garantir mesmo tamanho
n = min(len(df_fixo), len(df_inc))

df_fixo = df_fixo.iloc[:n]
df_inc = df_inc.iloc[:n]

print("===== MÉDIAS =====")

print("Fixo:")
print(df_fixo[["correctness","completeness","relevance","consistency","final_score"]].mean())

print("\nIncremental:")
print(df_inc[["correctness","completeness","relevance","consistency","final_score"]].mean())

cols = ["correctness","completeness","relevance","consistency","final_score"]

df_diff = df_inc[cols] - df_fixo[cols]

print("\n===== DIFERENÇA (Incremental - Fixo) =====")
print(df_diff.mean())

diff_media = (df_inc["final_score"] - df_fixo["final_score"]).mean()
print("Diferença média:", diff_media)

idx                    int64
pergunta              object
resposta_gabarito     object
resposta_gerada       object
correctness            int64
completeness           int64
relevance              int64
consistency            int64
final_score          float64
justification         object
dtype: object
idx                    int64
pergunta              object
resposta_gabarito     object
resposta_gerada       object
correctness            int64
completeness           int64
relevance              int64
consistency            int64
final_score          float64
justification         object
dtype: object
===== MÉDIAS =====
Fixo:
correctness     4.631579
completeness    4.236842
relevance       4.821053
consistency     4.715789
final_score     4.601316
dtype: float64

Incremental:
correctness     4.478947
completeness    3.952632
relevance       4.905263
consistency     4.600000
final_score     4.484211
dtype: float64

===== DIFERENÇA (Incremental - Fixo) =====
correctness    -0.152632
comp